# 🔍 Analiza danych — System detekcji podejrzanych transakcji

Notebook eksploracyjny do analizy alertów wygenerowanych przez pipeline:
**Producer → Kafka → Spark Streaming → CSV**

Bazuje na materiałach z Ćwiczeń 1-3 (scoring regułowy, ML, Isolation Forest).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Biblioteki załadowane.')

## 1. Wczytanie danych alertów

In [ ]:
# Spark zapisuje CSV jako katalog z plikami part-*
CSV_DIR = '../data/alerts.csv'

all_files = glob.glob(os.path.join(CSV_DIR, '**', 'part-*.csv'), recursive=True)
print(f'Znaleziono {len(all_files)} plików CSV')

if all_files:
    df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['processed_at'] = pd.to_datetime(df['processed_at'], errors='coerce')
    print(f'Wczytano {len(df)} alertów')
    df.head()

## 2. Statystyki opisowe

In [ ]:
print('=== Statystyki alertów ===')
print(f'Łączna liczba alertów: {len(df)}')
print(f'Średnia kwota: {df["amount"].mean():.2f} PLN')
print(f'Mediana kwoty: {df["amount"].median():.2f} PLN')
print(f'Max kwota: {df["amount"].max():.2f} PLN')
print()
print('=== Rozkład poziomu ryzyka ===')
print(df['risk_level'].value_counts())
print()
print('=== Rozkład kategorii ===')
print(df['category'].value_counts())

## 3. Wizualizacje

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1 Histogram kwot
axes[0, 0].hist(df['amount'], bins=30, color='coral', edgecolor='black')
axes[0, 0].set_title('Rozkład kwot alertów')
axes[0, 0].set_xlabel('Kwota (PLN)')
axes[0, 0].set_ylabel('Liczba')

# 3.2 Alerty per sklep
df['store'].value_counts().plot.bar(ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Alerty per sklep')
axes[0, 1].set_ylabel('Liczba alertów')

# 3.3 Alerty per kategoria
df['category'].value_counts().plot.pie(ax=axes[1, 0], autopct='%1.1f%%')
axes[1, 0].set_title('Alerty per kategoria')
axes[1, 0].set_ylabel('')

# 3.4 Risk score distribution
df['risk_score'].value_counts().sort_index().plot.bar(ax=axes[1, 1], color='tomato')
axes[1, 1].set_title('Rozkład risk_score')
axes[1, 1].set_xlabel('Risk Score')
axes[1, 1].set_ylabel('Liczba')

plt.tight_layout()
plt.savefig('../docs/analysis_charts.png', dpi=100)
plt.show()
print('Wykresy zapisane do docs/analysis_charts.png')

## 4. Analiza godzinowa

Sprawdzamy, o których godzinach występuje najwięcej podejrzanych transakcji.

In [ ]:
plt.figure(figsize=(12, 5))
hourly = df.groupby('hour').size()
hourly.plot.bar(color=['red' if h < 6 else 'steelblue' for h in hourly.index])
plt.title('Alerty per godzina (czerwone = godziny nocne)')
plt.xlabel('Godzina')
plt.ylabel('Liczba alertów')
plt.tight_layout()
plt.show()

## 5. Korelacja cech

In [ ]:
# Korelacja między cechami numerycznymi
numeric_cols = ['amount', 'hour', 'risk_score']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Macierz korelacji')
plt.tight_layout()
plt.show()

## 6. Podsumowanie

Pipeline skutecznie wykrywa podejrzane transakcje na podstawie prostych reguł biznesowych.
Główne cechy podejrzanych transakcji:
- **Wysokie kwoty** (>3000 PLN) — szczególnie w kategorii elektronika
- **Godziny nocne** (0-5) — nietypowa aktywność
- **Kombinacja cech** — elektronika + wysoka kwota + noc = najwyższe ryzyko